# Chapter 4: Information Theory

Companion notebook for *The Math That Powers AI* (2nd ed.), Chapter 4.

We use the `mathpowersai.information` module, which implements self-information,
entropy, KL divergence, cross-entropy, mutual information, binary symmetric channel
capacity, and perplexity exactly as developed in the chapter.

Conventions (from the chapter):

- Distributions are 1-D arrays of non-negative probabilities summing to 1.
- $0 \log 0 = 0$ (zero-probability terms are masked out).
- `base=2` gives **bits**; `base=np.e` gives **nats**.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np

from mathpowersai.information import (
    self_information,
    entropy,
    binary_entropy,
    kl_divergence,
    cross_entropy,
    mutual_information,
    bsc_capacity,
    perplexity,
)

## 1. Self-Information and the Entropy of Coin Flips

The **self-information** (surprisal) of an event with probability $p$ is

$$I(x) = -\log_2 p(x) \text{ bits.}$$

Rare events are surprising (high information); certain events carry none.
**Entropy** is the average surprisal over a whole distribution:

$$H(X) = -\sum_x p(x) \log_2 p(x).$$

A fair coin is the maximally uncertain binary source: exactly **1 bit** per flip.
A biased coin is more predictable, so each flip carries less information, and a
certain outcome carries none at all. A uniform distribution over 8 outcomes needs
$\log_2 8 = 3$ bits.

In [ ]:
# Self-information (surprisal), in bits
print(f"Fair coin heads (p=0.5): {self_information(0.5):.3f} bits")
print(f"Fair die six (p=1/6):    {self_information(1 / 6):.3f} bits")
print(f"Certain event (p=1.0):   {self_information(1.0):.3f} bits")

# Entropy of coin flips
print()
print(f"Fair coin (p=0.5): {entropy([0.5, 0.5]):.3f} bits")
print(f"Biased (p=0.9):    {entropy([0.9, 0.1]):.3f} bits")
print(f"Certain (p=1.0):   {entropy([1.0, 0.0]):.3f} bits")
print(f"Uniform over 8:    {entropy([1 / 8] * 8):.3f} bits")

assert np.isclose(entropy([0.5, 0.5]), 1.0)  # fair coin = exactly 1 bit

## 2. The Binary Entropy Curve

The **binary entropy function**

$$H(p) = -p \log_2 p - (1 - p) \log_2 (1 - p)$$

traces how the uncertainty of a coin varies with its bias. It is symmetric about
$p = 0.5$ (where it peaks at 1 bit) and falls to 0 at $p = 0$ and $p = 1$. Tabulating
a few values makes the shape clear without a plot.

In [ ]:
print(" p     H(p) [bits]")
for p in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    print(f" {p:.1f}   {binary_entropy(p):.4f}")

# Symmetry and the 1-bit maximum at p = 0.5
assert np.isclose(binary_entropy(0.3), binary_entropy(0.7))
assert np.isclose(binary_entropy(0.5), 1.0)
assert binary_entropy(0.0) == 0.0 and binary_entropy(1.0) == 0.0

## 3. KL Divergence Is Asymmetric

The **Kullback-Leibler divergence**

$$\mathrm{KL}(p \,\|\, q) = \sum_x p(x) \log_2 \frac{p(x)}{q(x)}$$

measures the expected *extra* bits paid when coding samples from the true
distribution $p$ with a code optimized for the model $q$. It is not a distance:
in general $\mathrm{KL}(p \,\|\, q) \ne \mathrm{KL}(q \,\|\, p)$.

Take a heavily biased true coin $p = (0.1, 0.9)$ and a uniform model
$q = (0.5, 0.5)$: the two directions give different numbers.

In [ ]:
p = [0.1, 0.9]  # True: biased toward outcome 2
q = [0.5, 0.5]  # Model: assumes uniform

kl_pq = kl_divergence(p, q)  # cost of using q when p is true
kl_qp = kl_divergence(q, p)  # cost of using p when q is true

print(f"KL(p || q) = {kl_pq:.4f} bits")
print(f"KL(q || p) = {kl_qp:.4f} bits")
print(f"Asymmetric: {not np.isclose(kl_pq, kl_qp)}")

# KL of a distribution with itself is zero
assert np.isclose(kl_divergence(p, p), 0.0)

## 4. Checkpoint: $H(p, q) = H(p) + \mathrm{KL}(p \,\|\, q)$

**Cross-entropy** decomposes into entropy plus KL divergence:

$$H(p, q) = -\sum_x p(x) \log_2 q(x) = H(p) + \mathrm{KL}(p \,\|\, q).$$

The total coding cost is the irreducible uncertainty of $p$ plus the penalty for
modeling it with $q$. We verify the identity for the chapter's checkpoint:
true distribution Bernoulli(0.7), model Bernoulli(0.5).

In [ ]:
p_b = [0.7, 0.3]  # Bernoulli(0.7)
q_b = [0.5, 0.5]  # Bernoulli(0.5)

h_p = entropy(p_b)
kl_pq = kl_divergence(p_b, q_b)
h_pq = cross_entropy(p_b, q_b)

print(f"H(p)              = {h_p:.4f} bits")
print(f"KL(p || q)        = {kl_pq:.4f} bits")
print(f"H(p, q)           = {h_pq:.4f} bits")
print(f"H(p) + KL(p || q) = {h_p + kl_pq:.4f} bits")

assert np.isclose(h_pq, h_p + kl_pq)
print("Identity holds: H(p, q) = H(p) + KL(p || q)")

## 5. Mutual Information and the BSC Capacity

**Mutual information** measures how many bits the output of a channel reveals about
its input:

$$I(X; Y) = \sum_{x,y} p(x, y) \log_2 \frac{p(x, y)}{p(x)\,p(y)} = H(Y) - H(Y \mid X).$$

For a **binary symmetric channel** (BSC) that flips each bit with probability
$p = 0.1$, the capacity (mutual information maximized by a uniform input) is

$$C = 1 - H(0.1) \approx 0.531 \text{ bits per symbol.}$$

We build the joint distribution for a uniform input through the BSC and confirm
that its mutual information equals the closed-form capacity.

In [ ]:
error_p = 0.1

# Joint p(x, y) for a uniform input through a BSC with crossover 0.1:
# p(x) = 0.5 for each input; correct transmission w.p. 0.9, flip w.p. 0.1.
joint = np.array([
    [0.5 * (1 - error_p), 0.5 * error_p],
    [0.5 * error_p, 0.5 * (1 - error_p)],
])
print("Joint p(x, y):")
print(joint)

mi = mutual_information(joint)
cap = bsc_capacity(error_p)
print()
print(f"H(0.1)                 = {binary_entropy(error_p):.3f} bits")
print(f"I(X; Y) from the joint = {mi:.3f} bits")
print(f"Capacity 1 - H(0.1)    = {cap:.3f} bits per symbol")

assert np.isclose(mi, cap)
assert np.isclose(cap, 0.531, atol=5e-4)

## 6. Perplexity from Cross-Entropy

Language models report **perplexity**, the exponentiated cross-entropy:

$$\mathrm{PPL} = e^{L} \quad (L \text{ in nats}), \qquad \mathrm{PPL} = 2^{H} \quad (H \text{ in bits}).$$

Both give the same number for the same quantity, since $2^{H_\text{bits}} = e^{H_\text{nats}}$.
A perplexity of $k$ means the model is as uncertain as choosing uniformly among $k$
equally likely words. We reuse the Bernoulli(0.7)/Bernoulli(0.5) pair from the
checkpoint.

In [ ]:
# Cross-entropy in nats (the default loss unit in frameworks like PyTorch)
ce_nats = cross_entropy(p_b, q_b, base=np.e)
ppl_nats = perplexity(ce_nats)  # base=e is the default
print(f"Cross-entropy = {ce_nats:.4f} nats -> perplexity = {ppl_nats:.4f}")

# Same quantity in bits gives the same perplexity
ce_bits = cross_entropy(p_b, q_b, base=2)
ppl_bits = perplexity(ce_bits, base=2)
print(f"Same in bits: 2**{ce_bits:.4f} = {ppl_bits:.4f}")

assert np.isclose(ppl_nats, ppl_bits)

# Sanity check: a uniform model over 2 outcomes has perplexity exactly 2
assert np.isclose(ppl_nats, 2.0)
print("A Bernoulli(0.5) model is as uncertain as a uniform choice among 2 outcomes.")

## Summary

- **Self-information** $-\log_2 p$ measures the surprise of a single event; a fair
  coin flip carries exactly **1 bit**.
- The **binary entropy curve** peaks at 1 bit for $p = 0.5$ and vanishes at the
  deterministic endpoints.
- **KL divergence** is the extra coding cost of a wrong model -- and it is
  **asymmetric**.
- **Cross-entropy** splits cleanly: $H(p, q) = H(p) + \mathrm{KL}(p \,\|\, q)$.
- **Mutual information** of a BSC with crossover 0.1 (uniform input) equals its
  capacity $1 - H(0.1) \approx 0.531$ bits per symbol.
- **Perplexity** is exponentiated cross-entropy, identical whether computed from
  nats or bits.